# RAG-applikation (Gym Exercises)
**Projekt i Datakvalitet och RAG**

I denna notebook bygger vi själva RAG-systemet (Retrieval-Augmented Generation). 
Vi utgår från vår städade högkvalitativa data (`megaGymDataset_ready.csv`). Eftersom vi i föregående steg tog bort rader med saknade instruktioner och dubbletter, säkerställer vi nu att AI-modellen får ren och relevant kontext att basera sina svar på.

In [2]:
# Importera nödvändiga bibliotek
import os
from dotenv import load_dotenv

# LangChain-komponenter för laddning och bearbetning
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma

# LangChain-komponenter för kedjor och prompter
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Ladda in API-nycklar från .env-filen (Viktigt att inte ladda upp på GitHub!)
load_dotenv()
#api_key = os.getenv("GEMINI_API_KEY")

True

#### 1. Ladda in den städade datan
Vi använder LangChains `CSVLoader` för att läsa in vår städade fil. Varje rad i CSV-filen kommer att omvandlas till ett eget "Document" som vår databas kan söka i.

In [3]:
# Sökvägen till din städade fil
file_path = 'megaGymDataset_ready.csv'

# Ladda in datan
loader = CSVLoader(file_path=file_path, encoding='utf-8')
documents = loader.load()

print(f"Laddade in {len(documents)} städade övningar från datasetet.")
# Skriv ut det första dokumentet för att verifiera att det ser rätt ut
print("\nExempel på hur ett dokument ser ut för AI:n:\n")
print(documents[0].page_content)

Laddade in 1359 städade övningar från datasetet.

Exempel på hur ett dokument ser ut för AI:n:

Title: Partner plank band row
Desc: The partner plank band row is an abdominal exercise where two partners perform single-arm planks while pulling on the opposite ends of an exercise band. This technique can be done for time or reps in any ab-focused workout.
Type: Strength
BodyPart: Abdominals
Equipment: Bands
Level: Intermediate


#### 2. Skapa Embeddings och Vektordatabas

För att AI:n ska kunna hitta rätt övning baserat på användarens fråga, måste vi översätta texten till siffror (vektorer). Vi använder Googles inbäddningsmodell och sparar resultatet i **ChromaDB**. Vi sparar databasen lokalt i mappen `chroma_db` så att vi inte behöver räkna om alla embeddings varje gång vi kör koden.

In [4]:
# Skapa vektordatabasen från vårt subset
# Mapp för att spara databasen lokalt
persist_directory = "./chroma_db"

# För att undvika 'rate limit' (Resource Exhausted) på gratis-API:et, 
# använder vi ett subset av datan, precis enligt kursens instruktioner.
subset = documents[:99]  # Skickar bara in de 99 första övningarna

print(f"Skapar embeddings för {len(subset)} dokument...")

# Skapa embeddings-instansen
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Skapa vektordatabasen från vårt subset
vector_store = Chroma.from_documents(
    documents=subset, 
    embedding=embeddings, 
    persist_directory=persist_directory
)

print("Vektordatabasen är skapad och redo för sökningar!")


Skapar embeddings för 99 dokument...
Vektordatabasen är skapad och redo för sökningar!


#### 3. Bygg RAG-kedjan (Retriever och LLM)
Nu kopplar vi ihop allt. Vi skapar en "Retriever" som hämtar de mest relevanta dokumenten från vår databas. Sedan instruerar vi LLM:en (Gemini) via en specifik **System Prompt** att agera som en personlig tränare och *enbart* svara utifrån den kontext vi skickar med.

In [5]:
# 1. Ställ in LLM 
# Vi använder temperature=0 för att göra modellen mer faktastyrd och mindre "kreativ" (undviker hallucinationer)
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

# 2. Skapa System Prompten
system_prompt = (
    "Du är en professionell och hjälpsam personlig tränare. "
    "Använd ENBART följande kontext för att svara på användarens fråga. "
    "Om svaret inte finns i kontexten, svara ärligt att du inte vet baserat på din nuvarande data, "
    "hitta inte på egna övningar.\n\n"
    "Kontext från vår databas:\n"
    "{context}"
)

# Sätt ihop prompten
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# 3. Skapa kedjan som hanterar dokumenten
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# 4. Gör om vektordatabasen till en retriever (hämtar de 3 mest relevanta övningarna)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 5. Koppla ihop retrievern och dokument-kedjan till den slutgiltiga RAG-kedjan
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

#### 4. Testa RAG-applikationen (Demonstration)
Här ställer vi en fråga till systemet. Systemet plockar fram en specifik övning från er CSV-fil och förklarar den i klartext.

In [12]:
# Test frågor för att se hur AI:n svarar baserat på den kontext den har från CSV-filen.
#question = "Hur utför man övningen 'FYR Banded Plank Jack'?"
#question = "Vilka övningar rekommenderar du för att träna 'rectus abdominis'."
question = "Ge mig ett exempel på en magövning (abdominals) jag kan göra tillsammans med min partner och vilka redskap som behövs"
#question = "Vilken är en bra övning för Triceps som använder E-Z Curl Bar? Beskriv hur man gör."
#question = "Hur gör man squats?"

# Skicka frågan genom kedjan
response = rag_chain.invoke({"input": question})

print(f"Fråga: {question}\n")
print(f"AI:ns Svar:\n{response['answer']}")

print("\n--------------------------------------------------\n")
print("Källor (Dokumenten som AI:n använde som kontext):")
# Skriv ut källorna för att bevisa att den tog det från CSV'n  
for i, doc in enumerate(response['context']):
    print(f"\nKälla {i+1}:")
    print(doc.page_content)

Fråga: Ge mig ett exempel på en magövning (abdominals) jag kan göra tillsammans med min partner och vilka redskap som behövs

AI:ns Svar:
En magövning du kan göra tillsammans med din partner är **Partner plank band row**.

Det är en styrkeövning för magmusklerna där ni båda står i en enarmad planka och drar i varsin ände av ett träningsband. Övningen kan utföras antingen på tid eller genom ett visst antal repetitioner.

**Redskap som behövs:**
* Träningsband (Bands)

--------------------------------------------------

Källor (Dokumenten som AI:n använde som kontext):

Källa 1:
Title: Partner plank band row
Desc: The partner plank band row is an abdominal exercise where two partners perform single-arm planks while pulling on the opposite ends of an exercise band. This technique can be done for time or reps in any ab-focused workout.
Type: Strength
BodyPart: Abdominals
Equipment: Bands
Level: Intermediate

Källa 2:
Title: Suspended ab fall-out
Desc: The suspended ab fall-out is a dynamic